# Bài tập 2 — Nguồn dữ liệu, cách thu thập và đánh giá chất lượng dữ liệu thô
**Môn:** Lập trình Phân tích Dữ liệu (2101681) · **Liên minh Nhóm 7 & 8**
**Phụ trách:** Nguyễn Bá Công, Hà Trọng Hữu Duy

| Nguồn | Vai trò | Cách thu thập | File trong `data/raw/` |
|---|---|---|---|
| nhadat.cafeland.vn | Chính | `requests` + BeautifulSoup, 12 luồng, checkpoint 100 tin (`00a_crawl_cafeland.ipynb`) | `cafeland_raw.csv` |
| batdongsan.vn | Bổ sung | Như trên (`00b_crawl_batdongsan.ipynb`) | `bat_dong_san_raw.xls` (thực chất là CSV UTF-8-sig) |
| chotot.com | Dự phòng — **chỉ khảo sát** | API JSON, `limit=100`, nghỉ 1–3 s | `chotot_bds_dataset.xls` (thực chất là CSV) |

Ngày cào: 31/08–01/09/2026. robots.txt của cả ba trang được kiểm tra ngày 01/09/2026.

In [1]:
import sys
from pathlib import Path

# Thư mục gốc dự án = thư mục cha của notebooks/
GOC = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(GOC))

import re
import numpy as np
import pandas as pd
from src.utils import doc_config

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
cfg = doc_config()
THU_MUC_THO = GOC / cfg['duong_dan']['du_lieu_tho']
THU_MUC_TRUNG_GIAN = GOC / cfg['duong_dan']['du_lieu_trung_gian']
THU_MUC_SACH = GOC / cfg['duong_dan']['du_lieu_sach']
print('Thư mục gốc:', GOC)

Thư mục gốc: /home/claude/repo


## 1. Các file dữ liệu thô

In [2]:
for f in sorted(THU_MUC_THO.glob('*')):
    if f.is_file() and f.name != '.gitkeep':
        print(f"  {f.name:<28} {f.stat().st_size/1024/1024:>7.2f} MB")

  bat_dong_san_raw.xls           51.30 MB
  cafeland_raw.csv                9.58 MB
  chotot_bds_dataset.xls          1.20 MB


## 2. Đọc từng nguồn ở dạng gốc (chưa ánh xạ)

In [3]:
cafe_goc = pd.read_csv(THU_MUC_THO / cfg['file_nguon']['cafeland'])
bds_goc = pd.read_csv(THU_MUC_THO / cfg['file_nguon']['batdongsan'], encoding='utf-8-sig')
chotot_goc = pd.read_csv(THU_MUC_THO / cfg['file_nguon']['chotot'])
for ten, d in [('CafeLand', cafe_goc), ('batdongsan', bds_goc), ('chotot', chotot_goc)]:
    print(f"{ten:<11} {d.shape[0]:>7,} dòng × {d.shape[1]} cột | {list(d.columns)}")

CafeLand      9,883 dòng × 9 cột | ['Tieu_de', 'Gia_ban', 'Dien_tich', 'Dia_diem', 'Loai_hinh_BDS', 'Mo_ta_Dac_diem', 'Nguoi_dang_Chu_dau_tu', 'Ngay_dang', 'URL']
batdongsan   30,149 dòng × 9 cột | ['Tieu_de', 'Gia_ban', 'Dien_tich', 'Dia_diem', 'Loai_hinh_BDS', 'Mo_ta_Dac_diem', 'Nguoi_dang_Chu_dau_tu', 'Ngay_dang', 'URL']
chotot        5,000 dòng × 14 cột | ['ad_id', 'title', 'price', 'area', 'price_per_m2', 'region_name', 'area_name', 'ward_name', 'street_name', 'property_type', 'rooms', 'direction', 'created_at', 'url']


In [4]:
cafe_goc.head(3)

,Tieu_de,Gia_ban,Dien_tich,Dia_diem,Loai_hinh_BDS,Mo_ta_Dac_diem,Nguoi_dang_Chu_dau_tu,Ngay_dang,URL
0,Môi giới nhà đất Nguyễn Thành Phước 0966755***,"4,X tỷ thương lượng. -","74,8m2",TP. Hồ Chí Minh,NaN,NaN,NaN,NaN,https://nhadat.cafeland.vn/moi-gioi/nguyen-tha...
1,Bán 2 lô đất biệt thự vườn có thể làm homestay...,3 tỷ 150 triệu,459m2,TP. Hồ Chí Minh,Biệt thự,Bán 2 lô đất biệt thự vườn có thể làm homestay...,(1 đánh giá) Đỗ Hồng Thắm Mobile:,25-08-2026,https://nhadat.cafeland.vn/ban-2-lo-dat-biet-t...
2,"Bán căn hộ 2PN 2WC 74,8m2 full NT The Sun Aven...",7 tỷ,74.8m2,TP. Hồ Chí Minh,Căn hộ,"Căn hộ chung cư tại The Sun Avenue, Mai Chí Th...",ngay,20-08-2026,https://nhadat.cafeland.vn/ban-can-ho-2pn-2wc-...


## 3. Cấu trúc biến và tỉ lệ thiếu theo cột

In [5]:
def bang_chat_luong(d):
    return pd.DataFrame({
        'kieu': d.dtypes.astype(str),
        'so_thieu': d.isna().sum(),
        'ty_le_thieu_%': (d.isna().mean() * 100).round(2),
        'so_gia_tri_khac_nhau': d.nunique(),
    })

print('CafeLand'); display(bang_chat_luong(cafe_goc))
print('batdongsan'); display(bang_chat_luong(bds_goc))

CafeLand


,kieu,so_thieu,ty_le_thieu_%,so_gia_tri_khac_nhau
Tieu_de,object,0,0.00,9538
Gia_ban,object,393,3.98,1739
Dien_tich,object,1224,12.38,1902
Dia_diem,object,0,0.00,1
Loai_hinh_BDS,object,421,4.26,21
Mo_ta_Dac_diem,object,422,4.27,9168
Nguoi_dang_Chu_dau_tu,object,433,4.38,2038
Ngay_dang,object,422,4.27,207
URL,object,0,0.00,9883


batdongsan


,kieu,so_thieu,ty_le_thieu_%,so_gia_tri_khac_nhau
Tieu_de,object,0,0.00,30130
Gia_ban,object,7,0.02,2297
Dien_tich,object,0,0.00,1271
Dia_diem,object,0,0.00,43
Loai_hinh_BDS,object,8355,27.71,11
Mo_ta_Dac_diem,object,0,0.00,30133
Nguoi_dang_Chu_dau_tu,object,0,0.00,2640
Ngay_dang,object,0,0.00,267
URL,object,0,0.00,30149


## 4. Trùng lặp và kiểu dữ liệu chưa nhất quán

In [6]:
for ten, d, cot_url in [('CafeLand', cafe_goc, 'URL'), ('batdongsan', bds_goc, 'URL'), ('chotot', chotot_goc, 'url')]:
    print(f"{ten:<11} trùng cả dòng: {d.duplicated().sum():>5,} | trùng URL: {d[cot_url].duplicated().sum():>5,}")

# Giá và diện tích đang là chuỗi chữ, nhiều cách viết
print()
print('Ví dụ Gia_ban:', cafe_goc['Gia_ban'].dropna().sample(6, random_state=1).tolist())
print('Ví dụ Dien_tich:', bds_goc['Dien_tich'].dropna().sample(6, random_state=1).tolist())
print('Số tin CafeLand ghi giá "Thương lượng":',
      cafe_goc['Gia_ban'].astype(str).str.contains('thương lượng', case=False).sum())

CafeLand    trùng cả dòng:     0 | trùng URL:     0
batdongsan  trùng cả dòng:     0 | trùng URL:     0
chotot      trùng cả dòng:     0 | trùng URL:     0

Ví dụ Gia_ban: ['7 tỷ', '4 tỷ 480 triệu', '4 tỷ 100 triệu', '6 tỷ', '8 tỷ 900 triệu', '3 tỷ 650 triệu']
Ví dụ Dien_tich: ['77 m²', '89 m²', '30 m²', '46 m²', '131 m²', '72 m²']
Số tin CafeLand ghi giá "Thương lượng": 490


In [7]:
# Chợ Tốt: vì sao không gộp
print('Tỉnh/thành:', chotot_goc['region_name'].value_counts().head(5).to_dict())
tieu_de = chotot_goc['title'].str.lower()
print('Tin có chữ "cho thuê":', tieu_de.str.contains('cho thuê').sum())
print('Loại hình:', chotot_goc['property_type'].value_counts().to_dict())

Tỉnh/thành: {'Tp Hồ Chí Minh': 3889, 'Đà Nẵng': 281, 'Bình Dương': 248, 'Hà Nội': 166, 'Đồng Nai': 95}
Tin có chữ "cho thuê": 1460
Loại hình: {'Nhà ở': 1954, 'Căn hộ/Chung cư': 1158, 'Phòng trọ': 937, 'Văn phòng, Mặt bằng kinh doanh': 504, 'Đất': 447}


## 5. Ánh xạ về schema chung (hàm trong `src/data/lam_sach_du_lieu.py`)

In [8]:
from src.data.lam_sach_du_lieu import doc_cafeland, doc_batdongsan
from src.utils import tom_tat_dataframe

df_cafeland = doc_cafeland(THU_MUC_THO / cfg['file_nguon']['cafeland'])
df_bds = doc_batdongsan(THU_MUC_THO / cfg['file_nguon']['batdongsan'])
for ten, d in [('CafeLand', df_cafeland), ('batdongsan', df_bds)]:
    tom_tat_dataframe(d, ten)

  [CafeLand]    đọc được 9,883 dòng


  [batdongsan]  đọc được 30,149 dòng

TÓM TẮT: CafeLand
Kích thước      : 9,883 dòng × 9 cột
Trùng lặp toàn bộ: 0 dòng
Thiếu dữ liệu   :
    quan_huyen                       9,883 (100.0%)
    dien_tich_m2                     1,224 ( 12.4%)
    gia_ban                            900 (  9.1%)
    mo_ta_dac_diem                     422 (  4.3%)
    loai_hinh                          421 (  4.3%)

TÓM TẮT: batdongsan
Kích thước      : 30,149 dòng × 9 cột


Trùng lặp toàn bộ: 0 dòng
Thiếu dữ liệu   :
    quan_huyen                      30,149 (100.0%)
    loai_hinh                        8,355 ( 27.7%)
    dien_tich_m2                        71 (  0.2%)
    gia_ban                             14 (  0.0%)


## 6. Nhận xét
- **Quy mô:** batdongsan.vn lớn nhất (30.149 tin), CafeLand 9.883 tin. CafeLand một mình chỉ còn ~3.500 tin tại TP.HCM sau làm sạch → chưa đủ 5.000, nên nhóm kích hoạt phương án dự phòng 1 (gộp batdongsan).
- **Chợ Tốt không gộp:** bộ 5.000 tin lẫn nhiều tỉnh và tin cho thuê, cột `area` bị lỗi khi cào (xem docstring `doc_chotot`). Hai nguồn còn lại đã đủ số dòng.
- **Chất lượng:** cả hai nguồn không có cột quận/huyện riêng; giá và diện tích là chuỗi chữ; batdongsan thiếu `Loai_hinh_BDS` khoảng 28%; CafeLand thiếu giá và diện tích ở một phần tin (bảng mục 3).
- **Bước tiếp theo:** `02_lam_sach_du_lieu.ipynb` quy đổi số, lọc TP.HCM, trích quận từ mô tả → `data/interim/df_raw.csv`.

### Ghi chú sử dụng công cụ AI
Theo mục IV của đề cương, nhóm ghi rõ phần có AI hỗ trợ:
- **Claude (Anthropic)** hỗ trợ: rà soát tính tái lập, gom các bước làm sạch thành hàm trong `src/`, đổi đường dẫn tuyệt đối sang tương đối, viết bảng kiểm toán số dòng bị loại, tái tạo bước lọc APE để kiểm chứng.
- Logic xử lý (ngưỡng lọc, cách điền khuyết, regex trích đặc trưng) do nhóm thiết kế và giải thích được; mọi con số trong notebook được sinh ra khi chạy lại, không nhập tay.